In [2]:
import altair as alt
import geopandas as gpd
import hvplot.pandas
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
pd.set_option("display.max_columns", None)

In [3]:
hoods = gpd.read_file("../data/raw/MUSA_zillow_neighborhoods.geojson").to_crs(epsg=2272)
parks = gpd.read_file("../data/raw/PPR_Properties.geojson").to_crs(epsg=2272)
streets = gpd.read_file("../data/raw/Street_Centerline.geojson").to_crs(epsg=2272)
city_limits = gpd.read_file("../data/raw/City_Limits.geojson").to_crs(epsg=2272)
viewpoints = gpd.read_file("../data/processed/viewpoints.geojson").to_crs(epsg=2272)

In [4]:
# strip unused columns from streets
clean_streets = streets.drop(columns=["zip_left", "zip_right", "length", "class", "responsibl", "update_", "oneway", "newsegdate", "multi_rep", "st_name", "st_type", "pre_dir", "suf_dir", "lpoly_", "rpoly_", "stcl2_", "stcl2_id", "streetlabe"])

# strip shit from parks
clean_parks = parks[["site_name", "park_name", "ppr_use", "geometry"]]
cleaner_streets = clean_streets[["stname", "seg_id", "geometry"]]


In [5]:
# Label viewpoints by which park/neighborhood they are in
viewpoint_hoods = gpd.sjoin(
    viewpoints,
    hoods,
    predicate="within",
    how="left",

)
viewpoint_hoods



,objectid,tree_name,scientific_name,common_name,Genus,Species,x,y,nearby_count,focal_species,geometry,index_right,ZillowName
0,54432,Platanus x acerifolia - london planetree,Platanus acerifolia,london planetree,Platanus,acerifolia,2.689582e+06,224063.610954,75,Platanus acerifolia,POINT (2689581.98 224063.611),55.0,Girard Estates
1,18693,Platanus x acerifolia - london planetree,Platanus acerifolia,london planetree,Platanus,acerifolia,2.690833e+06,238876.407469,66,Platanus acerifolia,POINT (2690833.226 238876.407),73.0,Logan Square
2,91255,Platanus x acerifolia - london planetree,Platanus acerifolia,london planetree,Platanus,acerifolia,2.701787e+06,268575.896586,62,Platanus acerifolia,POINT (2701787.411 268575.897),99.0,Olney
3,317,Platanus x acerifolia - london planetree,Platanus acerifolia,london planetree,Platanus,acerifolia,2.680540e+06,234459.213052,61,Platanus acerifolia,POINT (2680540.476 234459.213),126.0,Spruce Hill
4,117770,Platanus x acerifolia - london planetree,Platanus acerifolia,london planetree,Platanus,acerifolia,2.727120e+06,265031.199585,55,Platanus acerifolia,POINT (2727120.141 265031.2),131.0,Tacony
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,21305,Carya cordiformis - bitternut hickory,Carya cordiformis,bitternut hickory,Carya,cordiformis,2.680397e+06,228028.171482,12,Carya cordiformis,POINT (2680397.429 228028.171),5.0,Bartram Village
887,34408,Cryptomeria japonica - japanese cedar,Cryptomeria japonica,japanese cedar,Cryptomeria,japonica,2.692457e+06,254097.583506,19,Cryptomeria japonica,POINT (2692456.843 254097.584),56.0,Glenwood
888,1958,Diospyros virginiana - common persimmon,Diospyros virginiana,common persimmon,Diospyros,virginiana,2.691332e+06,260512.651973,9,Diospyros virginiana,POINT (2691332.47 260512.652),51.0,Germantown Southwest
889,32169,Cedrus deodara - deodar cedar,Cedrus deodara,deodar cedar,Cedrus,deodara,2.691055e+06,238948.446387,8,Cedrus deodara,POINT (2691055.389 238948.446),73.0,Logan Square


In [6]:
# viewpoint locs defines where the viewpoints are by park/neighborhood
viewpoint_locs = viewpoint_hoods.drop(columns="index_right")
viewpoint_locs = gpd.sjoin(
    viewpoint_locs,
    clean_parks,
    predicate="intersects",
    how="left"
)


In [7]:
viewpoint_locs = viewpoint_locs.drop(columns="index_right")
viewpoint_locs = gpd.sjoin_nearest(
    viewpoint_locs,
    clean_streets,
    max_distance=50,
    distance_col="dist_to_street",
    how="left"
)
viewpoint_locs

,objectid_left,tree_name,scientific_name,common_name,Genus,Species,x,y,nearby_count,focal_species,geometry,ZillowName,site_name,park_name,ppr_use,index_right,fnode_,tnode_,l_f_add,l_t_add,r_f_add,r_t_add,st_code,l_hundred,r_hundred,seg_id,stname,objectid_right,Shape__Length,dist_to_street
0,54432,Platanus x acerifolia - london planetree,Platanus acerifolia,london planetree,Platanus,acerifolia,2.689582e+06,224063.610954,75,Platanus acerifolia,POINT (2689581.98 224063.611),Girard Estates,NaN,NaN,NaN,869.0,25311.0,25425.0,2501.0,2599.0,2500.0,2598.0,88160.0,2500.0,2500.0,220066.0,S 19TH ST,870.0,181.981008,18.648432
1,18693,Platanus x acerifolia - london planetree,Platanus acerifolia,london planetree,Platanus,acerifolia,2.690833e+06,238876.407469,66,Platanus acerifolia,POINT (2690833.226 238876.407),Logan Square,Barnes Foundation Museum,Benjamin Franklin Parkway,MUSEUM_CAMPUS,10972.0,20240.0,20114.0,0.0,0.0,2001.0,2099.0,16880.0,0.0,2000.0,420751.0,BENJAMIN FRANKLIN PKWY,10973.0,260.583664,39.409836
2,91255,Platanus x acerifolia - london planetree,Platanus acerifolia,london planetree,Platanus,acerifolia,2.701787e+06,268575.896586,62,Platanus acerifolia,POINT (2701787.411 268575.897),Olney,NaN,NaN,NaN,30198.0,7743.0,7399.0,5800.0,5898.0,5801.0,5899.0,33240.0,5800.0,5800.0,740670.0,N FAIRHILL ST,30199.0,251.551805,15.346449
3,317,Platanus x acerifolia - london planetree,Platanus acerifolia,london planetree,Platanus,acerifolia,2.680540e+06,234459.213052,61,Platanus acerifolia,POINT (2680540.476 234459.213),Spruce Hill,Clarence H Clark Park,Clarence H Clark Park,NEIGHBORHOOD_PARK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,117770,Platanus x acerifolia - london planetree,Platanus acerifolia,london planetree,Platanus,acerifolia,2.727120e+06,265031.199585,55,Platanus acerifolia,POINT (2727120.141 265031.2),Tacony,NaN,NaN,NaN,31861.0,9440.0,9207.0,7000.0,7098.0,7001.0,7099.0,28660.0,7000.0,7000.0,760299.0,DITMAN ST,31862.0,284.017958,19.639341
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,21305,Carya cordiformis - bitternut hickory,Carya cordiformis,bitternut hickory,Carya,cordiformis,2.680397e+06,228028.171482,12,Carya cordiformis,POINT (2680397.429 228028.171),Bartram Village,Bartrams Garden,Bartrams Garden,HISTORIC_SITE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
887,34408,Cryptomeria japonica - japanese cedar,Cryptomeria japonica,japanese cedar,Cryptomeria,japonica,2.692457e+06,254097.583506,19,Cryptomeria japonica,POINT (2692456.843 254097.584),Glenwood,Vincent G Panati Playground,Vincent G Panati Playground,RECREATION_SITE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
888,1958,Diospyros virginiana - common persimmon,Diospyros virginiana,common persimmon,Diospyros,virginiana,2.691332e+06,260512.651973,9,Diospyros virginiana,POINT (2691332.47 260512.652),Germantown Southwest,Fernhill Park,Fernhill Park,RECREATION_SITE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
889,32169,Cedrus deodara - deodar cedar,Cedrus deodara,deodar cedar,Cedrus,deodara,2.691055e+06,238948.446387,8,Cedrus deodara,POINT (2691055.389 238948.446),Logan Square,Barnes Foundation Museum,Benjamin Franklin Parkway,MUSEUM_CAMPUS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
st_viewpoints = viewpoint_locs.loc[~viewpoint_locs["index_right"].isna()]
st_viewpoints.shape

(607, 30)

In [9]:
clean_viewpoint_locs = viewpoint_locs.drop(columns=["index_right"])
clean_viewpoint_locs.to_crs(epsg=4326).to_file("../data/processed/viewpoint_locations.geojson")



In [10]:
street_viewpoints = clean_viewpoint_locs.loc[~clean_viewpoint_locs["stname"].isna()]
street_viewpoints.shape

(607, 29)

In [11]:
cleaner_streets.to_file("../data/processed/streets.geojson")
clean_parks.to_file("../data/processed/parks.geojson")


In [16]:
streets_3857 = streets.to_crs(epsg=3857)
streets_3857

,fnode_,tnode_,lpoly_,rpoly_,length,stcl2_,stcl2_id,pre_dir,st_name,st_type,suf_dir,zip_left,zip_right,l_f_add,l_t_add,r_f_add,r_t_add,st_code,l_hundred,r_hundred,seg_id,oneway,class,responsibl,update_,newsegdate,multi_rep,streetlabe,stname,objectid,Shape__Length,geometry
0,26705,26716,None,None,2862.844615,None,None,,BARTRAM,AVE,,19153,19153,9200,9298,9201.0,9299.0,16120,9200,9200,100002,B,2,STATE,2004-05-18 00:00:00+00:00,NaT,0.0,BARTRAM AVE,BARTRAM AVE,1,1137.344551,"LINESTRING (-8377195.161 4848713.615, -8377217..."
1,26699,26704,None,None,935.023163,None,None,,BARTRAM,AVE,,19153,19153,9100,9198,9101.0,9199.0,16120,9100,9100,100003,B,2,STATE,1998-07-09 00:00:00+00:00,NaT,0.0,BARTRAM AVE,BARTRAM AVE,2,371.888030,"LINESTRING (-8376957.324 4849060.494, -8377166..."
2,26704,26705,None,None,122.512608,None,None,,BARTRAM,AVE,,19153,19153,0,0,0.0,0.0,16120,0,0,100004,B,2,STATE,1998-07-09 00:00:00+00:00,NaT,0.0,BARTRAM AVE,BARTRAM AVE,3,48.720227,"LINESTRING (-8377166.445 4848752.973, -8377195..."
3,26658,26672,None,None,735.818883,None,None,,EASTWICK,PL,,19153,19153,8500,8598,8501.0,8599.0,30570,8500,8500,100006,B,5,CITY,1998-07-09 00:00:00+00:00,NaT,0.0,EASTWICK PL,EASTWICK PL,4,292.693856,"LINESTRING (-8376319.145 4850300.173, -8376484..."
4,26653,26668,None,None,735.209140,None,None,,HARLEY,PL,,19153,19153,8500,8598,8501.0,8599.0,40570,8500,8500,100007,B,5,CITY,2006-04-07 00:00:00+00:00,NaT,0.0,HARLEY PL,HARLEY PL,5,292.440377,"LINESTRING (-8376399.383 4850357.154, -8376567..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41266,29256,29259,None,None,159.663507,None,None,,SUNFLOWER,DR,,19116,19116,15148,15152,15149.0,15153.0,75477,15100,15100,1180102,B,5,CITY,2011-08-30 00:00:00+00:00,2008-02-28 00:00:00+00:00,0.0,SUNFLOWER DR,SUNFLOWER DR,41267,63.661475,"LINESTRING (-8347313.106 4883597.986, -8347264..."
41267,29259,29258,None,None,218.496317,None,None,,MIMOSA,DR,,19116,19116,2601,2699,2600.0,2698.0,56360,2600,2600,1180103,B,5,CITY,2011-08-30 00:00:00+00:00,2008-02-28 00:00:00+00:00,0.0,MIMOSA DR,MIMOSA DR,41268,87.173485,"LINESTRING (-8347264.935 4883639.608, -8347211..."
41268,29259,29260,None,None,228.896914,None,None,,SUNFLOWER,DR,,19116,19116,15154,15198,15155.0,15199.0,75477,15100,15100,1180104,B,5,CITY,2011-08-30 00:00:00+00:00,2008-02-28 00:00:00+00:00,0.0,SUNFLOWER DR,SUNFLOWER DR,41269,91.299361,"LINESTRING (-8347264.935 4883639.608, -8347205..."
41269,29261,29260,None,None,390.822828,None,None,,BLUEBELL,CT,,19116,19116,2601,2699,2600.0,2698.0,18050,2600,2600,1180105,B,5,CITY,2011-08-30 00:00:00+00:00,2008-02-28 00:00:00+00:00,0.0,BLUEBELL CT,BLUEBELL CT,41270,155.870463,"LINESTRING (-8347309.233 4883763.873, -8347300..."
